# 01 — Baseline Model Training (ScanAI / VetVision AI)Trains the first working skin-disease (and secondary feature) classifieron the leakage-safe splits produced by `dataset_manager/split_dataset.py`.**Goal of this notebook:** verify the whole pipeline end-to-end and establishbaseline metrics — not to be the final model. Re-run this as your datasetgrows (see `main.py`'s "train early, iterate" workflow).Run this in Google Colab with a GPU runtime (Runtime → Change runtime type → GPU).

## 1. Imports

In [ ]:
import osimport jsonimport randomimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsimport tensorflow as tffrom tensorflow import kerasfrom tensorflow.keras import layersfrom tensorflow.keras.applications import EfficientNetB0from tensorflow.keras.applications.efficientnet import preprocess_inputfrom sklearn.metrics import (    confusion_matrix, classification_report, roc_curve, auc)from sklearn.preprocessing import label_binarizefrom sklearn.utils import class_weightSEED = 42random.seed(SEED)np.random.seed(SEED)tf.random.set_seed(SEED)print("TensorFlow:", tf.__version__)print("GPUs available:", tf.config.list_physical_devices('GPU'))

> **Reproducibility note:** `SEED = 42` above fixes Python's `random`, NumPy, and> TensorFlow's random state in one place. Every run in this notebook — data> shuffling, augmentation randomness, weight initialization — uses this same> seed. If you change it, log the new value in `docs/experiment_log.csv`> (see the logging cell near the end) so results stay traceable.

## 2. ConfigEdit `PROJECT_ROOT` if you mounted Google Drive at a different path, or ifyou're running locally instead of Colab.

In [ ]:
# If running in Colab, uncomment to mount Drive where your dataset_manager/splits/ lives:# from google.colab import drive# drive.mount('/content/drive')# PROJECT_ROOT = '/content/drive/MyDrive/ScanAI'PROJECT_ROOT = '.'  # change this to your actual ScanAI/ pathSPLITS_DIR = f'{PROJECT_ROOT}/dataset_manager/splits'MODELS_DIR = f'{PROJECT_ROOT}/models'OUTPUTS_DIR = f'{PROJECT_ROOT}/outputs'os.makedirs(MODELS_DIR, exist_ok=True)os.makedirs(f'{OUTPUTS_DIR}/confusion_matrix', exist_ok=True)os.makedirs(f'{OUTPUTS_DIR}/reports', exist_ok=True)os.makedirs(f'{OUTPUTS_DIR}/metrics', exist_ok=True)IMG_SIZE = (224, 224)BATCH_SIZE = 32EPOCHS_HEAD = 12       # phase 1: train the new classification head onlyEPOCHS_FINETUNE = 10   # phase 2: unfreeze top layers, fine-tune at low LRLR_HEAD = 1e-3LR_FINETUNE = 1e-5

## 3. Load the cleaned, split datasetLoads from `dataset_manager/splits/{train,val,test}/` — the leakage-safeoutput of `split_dataset.py`. Each subfolder name is a unified class from`taxonomy.json` (e.g. `skin__ringworm`, `parasites__tick_infestation`).

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(    f'{SPLITS_DIR}/train',    image_size=IMG_SIZE,    batch_size=BATCH_SIZE,    label_mode='categorical',    seed=SEED,)val_ds = tf.keras.utils.image_dataset_from_directory(    f'{SPLITS_DIR}/val',    image_size=IMG_SIZE,    batch_size=BATCH_SIZE,    label_mode='categorical',    seed=SEED,)test_ds = tf.keras.utils.image_dataset_from_directory(    f'{SPLITS_DIR}/test',    image_size=IMG_SIZE,    batch_size=BATCH_SIZE,    label_mode='categorical',    shuffle=False,   # keep order stable for confusion matrix / error analysis later    seed=SEED,)CLASS_NAMES = train_ds.class_namesNUM_CLASSES = len(CLASS_NAMES)print(f'{NUM_CLASSES} classes:')for c in CLASS_NAMES:    print(' -', c)

## 4. Visualise class distribution\n\nDo this *before* training — thin classes here explain weak per-class metrics later, not a training bug.

In [ ]:
def count_images(split_dir):    counts = {}    for class_name in sorted(os.listdir(split_dir)):        class_path = os.path.join(split_dir, class_name)        if os.path.isdir(class_path):            counts[class_name] = len([                f for f in os.listdir(class_path)                if f.lower().endswith(('.jpg', '.jpeg', '.png'))            ])    return countstrain_counts = count_images(f'{SPLITS_DIR}/train')df_counts = pd.DataFrame(list(train_counts.items()), columns=['class', 'train_count'])df_counts = df_counts.sort_values('train_count')plt.figure(figsize=(10, max(4, len(df_counts) * 0.3)))sns.barplot(data=df_counts, y='class', x='train_count', color='#4C72B0')plt.axvline(200, color='red', linestyle='--', label='minimum viable (200)')plt.title('Training images per class')plt.legend()plt.tight_layout()plt.savefig(f'{OUTPUTS_DIR}/reports/class_distribution.png', dpi=150)plt.show()print(df_counts.to_string(index=False))

## 5. Train/validation/test splitAlready handled by `split_dataset.py` (70/15/15, duplicate-cluster-aware —see that script's docstring for why this matters). This cell just confirmsthe split sizes look sane before you spend GPU time training.

In [ ]:
def count_total(ds):    return sum(x.shape[0] for x, y in ds)print('Train batches:', tf.data.experimental.cardinality(train_ds).numpy())print('Val batches:  ', tf.data.experimental.cardinality(val_ds).numpy())print('Test batches: ', tf.data.experimental.cardinality(test_ds).numpy())

## 6. PreprocessingUses EfficientNet's matched `preprocess_input` (scales to what theImageNet-pretrained weights expect) rather than a generic /255 rescale.

In [ ]:
AUTOTUNE = tf.data.AUTOTUNEdef preprocess(image, label):    image = preprocess_input(image)    return image, labeltrain_ds_prep = train_ds.map(preprocess, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)val_ds_prep = val_ds.map(preprocess, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)test_ds_prep = test_ds.map(preprocess, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

## 7. Data augmentationGeometric augmentation (flip/rotate/zoom) is applied broadly. **Color/contrastjitter is kept mild** — heavy color augmentation risks washing out theredness/discoloration that's often the actual diagnostic signal for skinconditions (see the project notes on this).

In [ ]:
data_augmentation = keras.Sequential([    layers.RandomFlip('horizontal'),    layers.RandomRotation(0.08),    layers.RandomZoom(0.1),    layers.RandomContrast(0.05),   # mild on purpose — see note above    layers.RandomTranslation(0.05, 0.05),], name='augmentation')

## 8. Build the model — EfficientNet-B0 transfer learningTwo-phase training:- **Phase 1:** freeze the ImageNet-pretrained backbone, train only the new  classification head.- **Phase 2 (fine-tune):** unfreeze the top ~20% of backbone layers, continue  training at a much lower learning rate.This is the standard, reliable recipe for small/medium datasets like this one.

In [ ]:
def build_model(num_classes, img_size):    base_model = EfficientNetB0(        include_top=False,        weights='imagenet',        input_shape=img_size + (3,),        pooling='avg',    )    base_model.trainable = False  # frozen for phase 1    inputs = keras.Input(shape=img_size + (3,))    x = data_augmentation(inputs)    x = base_model(x, training=False)    x = layers.Dropout(0.3)(x)    x = layers.Dense(256, activation='relu')(x)    x = layers.Dropout(0.2)(x)    outputs = layers.Dense(num_classes, activation='softmax')(x)    model = keras.Model(inputs, outputs)    return model, base_modelmodel, base_model = build_model(NUM_CLASSES, IMG_SIZE)model.summary()

## 8a. Class weightsSeveral of your classes (parasites, ear, eye) are much thinner than skindisease — class weighting stops the model from just always predicting thebiggest classes.

In [ ]:
# Derive integer labels from the training set for class_weight computationtrain_labels = []for class_name, count in train_counts.items():    train_labels.extend([CLASS_NAMES.index(class_name)] * count)weights = class_weight.compute_class_weight(    class_weight='balanced',    classes=np.arange(NUM_CLASSES),    y=np.array(train_labels),)class_weights = dict(enumerate(weights))print('Class weights (higher = rarer class, weighted more):')for i, cname in enumerate(CLASS_NAMES):    print(f'  {cname}: {class_weights[i]:.2f}')

## 9. Train — Phase 1 (frozen backbone)

In [ ]:
model.compile(    optimizer=keras.optimizers.Adam(learning_rate=LR_HEAD),    loss='categorical_crossentropy',    metrics=['accuracy', keras.metrics.AUC(name='auc', multi_label=True)],)callbacks_phase1 = [    keras.callbacks.ModelCheckpoint(        f'{MODELS_DIR}/baseline_phase1_best.keras',        save_best_only=True, monitor='val_accuracy', mode='max'    ),    keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2),]history_phase1 = model.fit(    train_ds_prep,    validation_data=val_ds_prep,    epochs=EPOCHS_HEAD,    class_weight=class_weights,    callbacks=callbacks_phase1,)

## 9a. Train — Phase 2 (fine-tune)Unfreezes the top layers of the backbone and continues training at a muchlower learning rate. Skip this cell if Phase 1 results already look strongenough for your timeline — fine-tuning adds real training time for often amodest accuracy gain.

In [ ]:
base_model.trainable = True# Freeze all but the last ~20% of layers — full unfreeze on a small dataset# tends to overfit badly.fine_tune_at = int(len(base_model.layers) * 0.8)for layer in base_model.layers[:fine_tune_at]:    layer.trainable = Falsemodel.compile(    optimizer=keras.optimizers.Adam(learning_rate=LR_FINETUNE),    loss='categorical_crossentropy',    metrics=['accuracy', keras.metrics.AUC(name='auc', multi_label=True)],)callbacks_phase2 = [    keras.callbacks.ModelCheckpoint(        f'{MODELS_DIR}/baseline_finetuned_best.keras',        save_best_only=True, monitor='val_accuracy', mode='max'    ),    keras.callbacks.EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),]history_phase2 = model.fit(    train_ds_prep,    validation_data=val_ds_prep,    epochs=EPOCHS_FINETUNE,    class_weight=class_weights,    callbacks=callbacks_phase2,)

## 10. Plot training curves

In [ ]:
def combine_histories(h1, h2, key):    return h1.history.get(key, []) + h2.history.get(key, [])acc = combine_histories(history_phase1, history_phase2, 'accuracy')val_acc = combine_histories(history_phase1, history_phase2, 'val_accuracy')loss = combine_histories(history_phase1, history_phase2, 'loss')val_loss = combine_histories(history_phase1, history_phase2, 'val_loss')fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))axes[0].plot(acc, label='train')axes[0].plot(val_acc, label='val')axes[0].axvline(len(history_phase1.history['accuracy']), color='gray', linestyle='--', label='fine-tune start')axes[0].set_title('Accuracy')axes[0].set_xlabel('Epoch')axes[0].legend()axes[1].plot(loss, label='train')axes[1].plot(val_loss, label='val')axes[1].axvline(len(history_phase1.history['loss']), color='gray', linestyle='--', label='fine-tune start')axes[1].set_title('Loss')axes[1].set_xlabel('Epoch')axes[1].legend()plt.tight_layout()plt.savefig(f'{OUTPUTS_DIR}/reports/training_curves.png', dpi=150)plt.show()

## 11. Evaluate on the held-out test set\n\nThis set was never touched during training or validation — this is the number that goes in your report.

In [ ]:
test_loss, test_acc, test_auc = model.evaluate(test_ds_prep)print(f'Test accuracy: {test_acc:.4f}')print(f'Test AUC:      {test_auc:.4f}')print(f'Test loss:     {test_loss:.4f}')

## 12. Confusion matrix

In [ ]:
y_true = []y_pred_probs = []for images, labels in test_ds_prep:    preds = model.predict(images, verbose=0)    y_pred_probs.extend(preds)    y_true.extend(np.argmax(labels.numpy(), axis=1))y_true = np.array(y_true)y_pred_probs = np.array(y_pred_probs)y_pred = np.argmax(y_pred_probs, axis=1)cm = confusion_matrix(y_true, y_pred)plt.figure(figsize=(max(8, NUM_CLASSES * 0.5), max(6, NUM_CLASSES * 0.5)))sns.heatmap(cm, annot=NUM_CLASSES <= 20, fmt='d', cmap='Blues',            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)plt.xlabel('Predicted')plt.ylabel('True')plt.title('Confusion Matrix — Test Set')plt.xticks(rotation=90)plt.yticks(rotation=0)plt.tight_layout()plt.savefig(f'{OUTPUTS_DIR}/confusion_matrix/test_confusion_matrix.png', dpi=150)plt.show()

## 13. Classification report (precision / recall / F1 per class)

In [ ]:
report_dict = classification_report(    y_true, y_pred, target_names=CLASS_NAMES, output_dict=True, zero_division=0)report_df = pd.DataFrame(report_dict).transpose()report_df.to_csv(f'{OUTPUTS_DIR}/metrics/classification_report.csv')print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))# Flag classes performing worst — direct input for 02_error_analysis.ipynbweak_classes = report_df.iloc[:NUM_CLASSES].sort_values('f1-score').head(5)print('\nWeakest classes by F1 (investigate these first in error analysis):')print(weak_classes[['precision', 'recall', 'f1-score', 'support']])

## 14. ROC / AUC (one-vs-rest, per class)

In [ ]:
y_true_bin = label_binarize(y_true, classes=list(range(NUM_CLASSES)))plt.figure(figsize=(9, 7))auc_scores = {}for i, class_name in enumerate(CLASS_NAMES):    if y_true_bin[:, i].sum() == 0:        continue  # class not present in test set, skip    fpr, tpr, _ = roc_curve(y_true_bin[:, i], y_pred_probs[:, i])    roc_auc = auc(fpr, tpr)    auc_scores[class_name] = roc_auc    if NUM_CLASSES <= 15:  # only plot individually if not too cluttered        plt.plot(fpr, tpr, label=f'{class_name} (AUC={roc_auc:.2f})')plt.plot([0, 1], [0, 1], 'k--', alpha=0.4)plt.xlabel('False Positive Rate')plt.ylabel('True Positive Rate')plt.title('ROC Curves — One-vs-Rest (Test Set)')if NUM_CLASSES <= 15:    plt.legend(fontsize=8, loc='lower right')plt.tight_layout()plt.savefig(f'{OUTPUTS_DIR}/reports/roc_curves.png', dpi=150)plt.show()auc_df = pd.DataFrame(list(auc_scores.items()), columns=['class', 'auc']).sort_values('auc')auc_df.to_csv(f'{OUTPUTS_DIR}/metrics/per_class_auc.csv', index=False)print(auc_df.to_string(index=False))

## 15. Save the model + log run metadataSaves in Keras format for now — TFLite/ONNX conversion happens in`04_export_model.ipynb`, after you've confirmed this baseline is worthdeploying.

In [ ]:
final_model_path = f'{MODELS_DIR}/baseline_efficientnet_b0.keras'model.save(final_model_path)# Model size + param count — you'll want these for your report's model comparison tablenum_params = model.count_params()model_size_mb = os.path.getsize(final_model_path) / (1024 * 1024)run_metadata = {    'model_architecture': 'EfficientNetB0 (transfer learning, 2-phase)',    'num_classes': NUM_CLASSES,    'class_names': CLASS_NAMES,    'img_size': IMG_SIZE,    'batch_size': BATCH_SIZE,    'epochs_head': EPOCHS_HEAD,    'epochs_finetune': EPOCHS_FINETUNE,    'test_accuracy': float(test_acc),    'test_auc': float(test_auc),    'test_loss': float(test_loss),    'num_params': int(num_params),    'model_size_mb': round(model_size_mb, 2),}with open(f'{OUTPUTS_DIR}/metrics/run_metadata.json', 'w') as f:    json.dump(run_metadata, f, indent=2)print(json.dumps(run_metadata, indent=2))print(f'\nModel saved to {final_model_path}')print('Next notebook: 02_error_analysis.ipynb')

## 16. Log this run to the experiment trackerAppends one row to `docs/experiment_log.csv` (created once, appended to onevery run — never overwritten). This is what lets you answer "which configgave the best F1?" without re-opening old notebooks.

In [ ]:
import csvfrom datetime import datetimeEXPERIMENT_LOG = f'{PROJECT_ROOT}/docs/experiment_log.csv'# Macro-averaged precision/recall/f1 across classes, for a single summary row# (per-class detail already lives in outputs/metrics/classification_report.csv)macro_precision = report_dict['macro avg']['precision']macro_recall = report_dict['macro avg']['recall']macro_f1 = report_dict['macro avg']['f1-score']macro_auc = float(np.mean(list(auc_scores.values()))) if auc_scores else float('nan')experiment_id = datetime.now().strftime('exp_%Y%m%d_%H%M%S')log_row = {    'experiment_id': experiment_id,    'date': datetime.now().strftime('%Y-%m-%d'),    'model': 'EfficientNetB0',    'learning_rate': f'{LR_HEAD} (head) / {LR_FINETUNE} (finetune)',    'batch_size': BATCH_SIZE,    'epochs': f'{EPOCHS_HEAD}+{EPOCHS_FINETUNE}',    'augmentation': 'flip+rotate+zoom+mild_contrast+translate',    'accuracy': round(float(test_acc), 4),    'precision': round(float(macro_precision), 4),    'recall': round(float(macro_recall), 4),    'f1': round(float(macro_f1), 4),    'auc': round(macro_auc, 4) if not np.isnan(macro_auc) else '',    'notes': '',  # fill in manually, e.g. "first baseline run" or "after adding tick data"}file_exists = os.path.exists(EXPERIMENT_LOG)os.makedirs(os.path.dirname(EXPERIMENT_LOG), exist_ok=True)with open(EXPERIMENT_LOG, 'a', newline='') as f:    writer = csv.DictWriter(f, fieldnames=list(log_row.keys()))    if not file_exists or os.path.getsize(EXPERIMENT_LOG) == 0:        writer.writeheader()    writer.writerow(log_row)print(f'Logged run {experiment_id} to {EXPERIMENT_LOG}')print(log_row)

## 17. External hold-out evaluation (optional — run once, near the end)Evaluates against `external_test/` — photos that **never passed through`dataset_manager/`'s pipeline at all** (no dedup, no augmentation, none ofyour training/val/test splits). This is your honest real-worldgeneralization check.**Only run this once you consider the model close to final.** Repeatedlytuning against this folder turns it into a second validation set anddefeats its purpose — see `external_test/README.md`.Skip this cell entirely if `external_test/` is still empty.

In [ ]:
EXTERNAL_TEST_DIR = f'{PROJECT_ROOT}/external_test'has_labeled_subfolders = (    os.path.isdir(EXTERNAL_TEST_DIR)    and any(        os.path.isdir(os.path.join(EXTERNAL_TEST_DIR, d)) and d != 'unlabeled'        and len(os.listdir(os.path.join(EXTERNAL_TEST_DIR, d))) > 0        for d in os.listdir(EXTERNAL_TEST_DIR)    ))if not has_labeled_subfolders:    print('external_test/ is empty or has no labeled class folders yet — skipping.')    print('Add photos under external_test/<class_name>/ using the same class names as taxonomy.json, then re-run this cell.')else:    external_ds = tf.keras.utils.image_dataset_from_directory(        EXTERNAL_TEST_DIR,        image_size=IMG_SIZE,        batch_size=BATCH_SIZE,        label_mode='categorical',        shuffle=False,    )    external_class_names = external_ds.class_names    external_ds_prep = external_ds.map(preprocess, num_parallel_calls=AUTOTUNE)    ext_loss, ext_acc, ext_auc = model.evaluate(external_ds_prep)    print(f'External hold-out accuracy: {ext_acc:.4f}')    print(f'External hold-out AUC:      {ext_auc:.4f}')    print()    print('Compare this to the internal test accuracy above.')    print('A large gap (external much lower than internal test) usually means')    print('the training data does not match real-world photo conditions well')    print('(lighting, camera quality, framing) — a stronger signal than internal')    print('test accuracy alone for how the app will actually perform.')    with open(f'{OUTPUTS_DIR}/metrics/external_holdout_result.json', 'w') as f:        json.dump({            'external_accuracy': float(ext_acc),            'external_auc': float(ext_auc),            'internal_test_accuracy': float(test_acc),            'gap': float(test_acc) - float(ext_acc),        }, f, indent=2)